<a href="https://colab.research.google.com/github/ClaudeVerlain/DSA-PROJECT/blob/main/enveda-casmi-2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation of RDKit and matchms

In [ ]:
!pip install rdkit matchms pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.8/213.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -q kaggle

### Configure the Kaggle token as an environment variable in Colab

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

### Download competition data

In [ ]:
%pip install -q --upgrade kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 2.8 MB/s eta 0:00:00


In [ ]:
!kaggle --version
#!kaggle competitions list

Kaggle CLI 2.2.4


### Download the data

In [ ]:
!kaggle competitions download \
    -c enveda-casmi26-molecule-id-mass-spectra \
    -p /content/data

100% 2.82G/2.82G [00:25<00:00, 120MB/s]



In [ ]:
!ls -lh /content/data

total 2.9G
-rw-r--r-- 1 root root 2.9G Sep 15 19:27 enveda-casmi26-molecule-id-mass-spectra.zip


### Extracted files

In [ ]:
!unzip -o /content/data/enveda-casmi26-molecule-id-mass-spectra.zip \
    -d /content/data/extracted

Archive:  /content/data/enveda-casmi26-molecule-id-mass-spectra.zip
  inflating: /content/data/extracted/sample_submission.csv  
  inflating: /content/data/extracted/test.parquet  
  inflating: /content/data/extracted/train.parquet  


### List of files

In [ ]:
import os

for root, dirs, files in os.walk("/content/data/extracted"):
    for file in files:
        path = os.path.join(root, file)
        print(f"{file} — {os.path.getsize(path) / (1024**2):.2f} MB")

sample_submission.csv — 0.04 MB
test.parquet — 4.62 MB
train.parquet — 2892.77 MB


### Find CSV files

In [ ]:
import glob
import pandas as pd

csv_files = glob.glob(
    "/content/data/extracted/**/*.csv",
    recursive=True
)

print("CSV files found:")
for i, path in enumerate(csv_files):
    print(i, path)

CSV files found:
0 /content/data/extracted/sample_submission.csv


### Show submission head

In [ ]:
submission_path = next(
    (p for p in csv_files if "sample_submission" in p.lower()),
    None
)

if submission_path:
    df = pd.read_csv(submission_path)
    display(df.head())
else:
    print("No sample_submission.csv file found.")

,molecule_id,smiles
0,m_005e53,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
1,m_006153,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
2,m_00b5aa,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
3,m_01e922,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
4,m_0259d4,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...


### View the file without loading it completely

In [ ]:
!pip install -q pyarrow

In [ ]:
import pyarrow.parquet as pq

train_path = "/content/data/extracted/train.parquet"
test_path = "/content/data/extracted/test.parquet"

for name, path in [("TRAIN", train_path), ("TEST", test_path)]:
    pf = pq.ParquetFile(path)

    print(f"\n--- {name} ---")
    print("Number of rows:", pf.metadata.num_rows)
    print("Number of columns:", len(pf.schema.names))
    print("Row groups:", pf.metadata.num_row_groups)
    print("Columns:", pf.schema.names)
    print("Schema:")
    print(pf.schema)


--- TRAIN ---
Number of rows: 2539608
Number of columns: 18
Row groups: 21
Columns: ['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'element', 'element', 'num_peaks', 'base_peak_intensity', 'element', 'collision_energy_orig', 'collision_energy_orig_units']
Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 ingest_lib (String);
  optional binary field_id=-1 normalized_smiles (String);
  optional binary field_id=-1 inchikey (String);
  optional binary field_id=-1 inchikey14 (String);
  optional binary field_id=-1 molecular_formula (String);
  optional binary field_id=-1 ionization_mode (String);
  optional binary field_id=-1 instrument_type (String);
  optional binary field_id=-1 adduct (String);
  optional binary field_id=-1 adduct_orig (String);
  optional double field_id=-1 precursor_mz;
  optional double field_id=-1 precursor

### Load a sample of the train dataset

In [ ]:
import pyarrow.parquet as pq

train_path = "/content/data/extracted/train.parquet"

pf = pq.ParquetFile(train_path)

# Load only 100 rows and selected columns
sample = next(
    pf.iter_batches(
        batch_size=100,
        columns=["inchikey", "adduct", "precursor_mz", "num_peaks"]
    )
).to_pandas()

print("Sample Dimensions:", sample.shape)
display(sample.head(10))

Sample Dimensions: (100, 4)


,inchikey,adduct,precursor_mz,num_peaks
0,AAALVYBICLMAMA-UHFFFAOYSA-N,[M+H]+,330.1237,52
1,AAKJLRGGTJKAMG-UHFFFAOYSA-N,[M+K]+,432.1320,28
2,AANFHDFOMFRLLR-UHFFFAOYSA-N,[M+Na]+,422.1738,41
3,AAOVKJBEBIDNHE-UHFFFAOYSA-N,[M+H]+,285.0789,166
4,ACFIXJIJDZMPPO-UHFFFAOYSA-N,[M+H]+,746.0984,186
5,ACGDKVXYNVEAGU-UHFFFAOYSA-N,[M+H]+,199.1917,119
6,ACNHBCIZLNNLRS-UHFFFAOYSA-N,[M+K]+,474.2041,200
7,ACPOUJIDANTYHO-UHFFFAOYSA-N,[M+H]+,221.0709,9
8,ACRHBAYQBXXRTO-UHFFFAOYSA-N,[M+K]+,507.2256,55
9,ACTIUHUUMQJHFO-UHFFFAOYSA-N,[M+NH4]+,863.6912,200


### General statistics calculation for the train set (without loading the entire file in memory)

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from collections import Counter

train_path = "/content/data/extracted/train.parquet"
test_path = "/content/data/extracted/test.parquet"

# Counters and accumulators
molecule_counts = Counter()
adduct_counts = Counter()
mass_arrays = []
peak_arrays = []

total_spectra = 0
missing_molecule = 0
missing_adduct = 0
missing_mass = 0

pf = pq.ParquetFile(train_path)

for i, batch in enumerate(
    pf.iter_batches(
        batch_size=100_000,
        columns=["inchikey", "adduct", "precursor_mz", "num_peaks"]
    )
):
    df = batch.to_pandas()
    total_spectra += len(df)

    # Number of spectra per molecule
    molecules = df["inchikey"].dropna().astype(str).value_counts()
    molecule_counts.update(molecules.to_dict())
    missing_molecule += int(df["inchikey"].isna().sum())

    # Adduct distribution
    adducts = df["adduct"].dropna().astype(str).value_counts()
    adduct_counts.update(adducts.to_dict())
    missing_adduct += int(df["adduct"].isna().sum())

    # Masses and peak counts
    masses = pd.to_numeric(df["precursor_mz"], errors="coerce").dropna()
    missing_mass += int(df["precursor_mz"].isna().sum())
    peaks = pd.to_numeric(df["num_peaks"], errors="coerce").dropna()

    mass_arrays.append(masses.to_numpy(dtype=np.float64))
    peak_arrays.append(peaks.to_numpy(dtype=np.float64))

    print(f"Batch {i + 1} processed — {total_spectra:,} spectra")

# Merge numeric arrays
masses = np.concatenate(mass_arrays) if mass_arrays else np.array([])
peaks = np.concatenate(peak_arrays) if peak_arrays else np.array([])

print("\n===== GENERAL STATISTICS =====")
print(f"Number of spectra: {total_spectra:,}")
print(f"Number of distinct molecules: {len(molecule_counts):,}")
print(f"Spectres without molecule identifier: {missing_molecule:,}")
print(f"Spectres without adduct: {missing_adduct:,}")
print(f"Missing precursor masses: {missing_mass:,}")

print("\n===== TOP 20 MOLECULES BY SPECTRA COUNT =====")
molecules_df = pd.DataFrame(
    molecule_counts.most_common(20),
    columns=["inchikey", "spectra_count"]
)
display(molecules_df)

print("\n===== ADDUCT DISTRIBUTION =====")
adduct_df = pd.DataFrame(
    adduct_counts.most_common(),
    columns=["adduct", "spectra_count"]
)
adduct_df["percentage"] = (
    100 * adduct_df["spectra_count"] / total_spectra
).round(2)
display(adduct_df)

print("\n===== PRECURSOR MASS DISTRIBUTION (precursor_mz) =====")
if len(masses):
    print(pd.Series(masses).describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ))

print("\n===== NUMBER OF PEAKS PER SPECTRUM =====")
if len(peaks):
    print(pd.Series(peaks).describe())

# Save statistics results
molecules_df.to_csv("/content/data/top_molecules.csv", index=False)
adduct_df.to_csv("/content/data/adduct_distribution.csv", index=False)

print("\nResults saved in /content/data/")

Batch 1 processed — 100,000 spectra
Batch 2 processed — 200,000 spectra
Batch 3 processed — 300,000 spectra
Batch 4 processed — 400,000 spectra
Batch 5 processed — 500,000 spectra
Batch 6 processed — 600,000 spectra
Batch 7 processed — 700,000 spectra
Batch 8 processed — 800,000 spectra
Batch 9 processed — 900,000 spectra
Batch 10 processed — 1,000,000 spectra
Batch 11 processed — 1,100,000 spectra
Batch 12 processed — 1,200,000 spectra
Batch 13 processed — 1,300,000 spectra
Batch 14 processed — 1,400,000 spectra
Batch 15 processed — 1,500,000 spectra
Batch 16 processed — 1,600,000 spectra
Batch 17 processed — 1,700,000 spectra
Batch 18 processed — 1,800,000 spectra
Batch 19 processed — 1,900,000 spectra
Batch 20 processed — 2,000,000 spectra
Batch 21 processed — 2,100,000 spectra
Batch 22 processed — 2,200,000 spectra
Batch 23 processed — 2,300,000 spectra
Batch 24 processed — 2,400,000 spectra
Batch 25 processed — 2,500,000 spectra
Batch 26 processed — 2,539,608 spectra

===== GENERA

,inchikey,spectra_count
0,PFTAWBLQPZVEMU-UHFFFAOYSA-N,1468
1,LPLVUJXQOOQHMX-UHFFFAOYSA-N,1329
2,AIONOLUJZLIMTK-UHFFFAOYSA-N,1303
3,RWXIFXNRCLMQCD-UHFFFAOYSA-N,1171
4,TZBJGXHYKVUXJN-UHFFFAOYSA-N,1018
5,KRKNYBCHXYNGOX-UHFFFAOYSA-N,941
6,OYHQOLUKZRVURQ-UHFFFAOYSA-N,917
7,REFJWTPEDVJJIY-UHFFFAOYSA-N,909
8,LOUPRKONTZGTKE-UHFFFAOYSA-N,905
9,OVSQVDMCBVZWGM-UHFFFAOYSA-N,877



===== ADDUCT DISTRIBUTION =====


,adduct,spectra_count,percentage
0,[M+H]+,1324585,52.16
1,[M-H]-,465688,18.34
2,[2M+Na]+,190033,7.48
3,[M+Na]+,178092,7.01
4,[2M+H]+,92041,3.62
...,...,...,...
116,[M-CH3O]+,1,0.00
117,[2M-H5O3]+,1,0.00
118,[M+C2H7OS]+,1,0.00
119,[3M+K]+,1,0.00



===== PRECURSOR MASS DISTRIBUTION (precursor_mz) =====
count    2.539608e+06
mean     4.197696e+02
std      1.535987e+03
min      2.000000e+00
25%      3.081410e+02
50%      3.501088e+02
75%      4.892142e+02
90%      6.994726e+02
95%      7.754660e+02
99%      9.763292e+02
max      2.430333e+06
dtype: float64

===== NUMBER OF PEAKS PER SPECTRUM =====
count    2.539608e+06
mean     1.582158e+02
std      5.168801e+02
min      1.000000e+00
25%      1.400000e+01
50%      4.200000e+01
75%      1.640000e+02
max      7.331800e+04
dtype: float64

Results saved in /content/data/


### Analysis of the test file

In [ ]:
import pandas as pd

test_path = "/content/data/extracted/test.parquet"

# Load test file: only the necessary columns
test_df = pd.read_parquet(
    test_path,
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "ionization_mode",
        "precursor_mz",
        "base_peak_intensity"
    ]
)

print("===== TEST DATA DIMENSIONS =====")
print(f"Number of spectra: {len(test_df):,}")
print(f"Number of unique molecules: {test_df['molecule_id'].nunique():,}")
print(f"Number of unique spectra: {test_df['spectrum_id'].nunique():,}")

print("\n===== SPECTRA PER MOLECULE =====")
display(
    test_df.groupby("molecule_id")
    .size()
    .describe()
    .to_frame("statistics")
)

print("\n===== ADDUCT DISTRIBUTION =====")
display(
    test_df["adduct"]
    .value_counts(dropna=False)
    .rename_axis("adduct")
    .reset_index(name="spectra_count")
)

print("\n===== IONIZATION MODES =====")
display(
    test_df["ionization_mode"]
    .value_counts(dropna=False)
    .rename_axis("ionization_mode")
    .reset_index(name="spectra_count")
)

print("\n===== PRECURSOR MASSES =====")
display(test_df["precursor_mz"].describe())

print("\n===== BASE PEAK INTENSITY =====")
display(test_df["base_peak_intensity"].describe())

print("\n===== MISSING VALUES =====")
display(test_df.isna().sum().to_frame("missing_values"))

test_df.to_csv("/content/data/test_basic_statistics.csv", index=False)
print("\nTest table saved in /content/data/test_basic_statistics.csv")

===== TEST DATA DIMENSIONS =====
Number of spectra: 1,213
Number of unique molecules: 400
Number of unique spectra: 1,213

===== SPECTRA PER MOLECULE =====


,statistics
count,400.000000
mean,3.032500
std,1.382469
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,9.000000



===== ADDUCT DISTRIBUTION =====


,adduct,spectra_count
0,[M+H]+,959
1,[M-H]-,193
2,[M+CH2O2-H]-,31
3,[M+Na]+,22
4,[M+NH4]+,4
5,[M+K]+,2
6,[M+Cl]-,2



===== IONIZATION MODES =====


...


===== BASE PEAK INTENSITY =====


,base_peak_intensity
count,1.213000e+03
mean,3.509205e+06
std,8.547499e+06
min,1.018000e+03
25%,8.689600e+04
50%,4.742270e+05
75%,2.768804e+06
max,1.123810e+08



===== MISSING VALUES =====


,missing_values
molecule_id,0
spectrum_id,0
adduct,0
ionization_mode,0
precursor_mz,0
base_peak_intensity,0



Test table saved in /content/data/test_basic_statistics.csv


### Implement spectrum cleanup

In [ ]:

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

train_path = "/content/data/extracted/train.parquet"

# Configurable parameters
MIN_RELATIVE_INTENSITY = 0.01  # Keep peaks with at least 1% of the maximum intensity
PRECURSOR_TOLERANCE_PPM = 10   # Precursor mass tolerance in ppm


def clean_spectrum(
    mzs,
    intensities,
    precursor_mz,
    min_relative_intensity=MIN_RELATIVE_INTENSITY,
    precursor_tolerance_ppm=PRECURSOR_TOLERANCE_PPM
):
    """Clean an MS/MS spectrum and return cleaned masses and intensities."""

    if mzs is None or intensities is None:
        return [], []

    mzs = np.asarray(mzs, dtype=float)
    intensities = np.asarray(intensities, dtype=float)

    # Keep mass and intensity arrays aligned
    n = min(len(mzs), len(intensities))
    mzs = mzs[:n]
    intensities = intensities[:n]

    # Remove invalid masses, invalid intensities, and negative intensities
    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities >= 0)
    )
    mzs = mzs[valid]
    intensities = intensities[valid]

    if len(mzs) == 0:
        return [], []

    # 1. Remove the precursor peak when its mass is available
    if precursor_mz is not None and pd.notna(precursor_mz):
        precursor_mz = float(precursor_mz)

        if np.isfinite(precursor_mz) and precursor_mz > 0:
            tolerance = max(
                0.005,
                precursor_mz * precursor_tolerance_ppm * 1e-6
            )
            keep = np.abs(mzs - precursor_mz) > tolerance
            mzs = mzs[keep]
            intensities = intensities[keep]

    if len(mzs) == 0:
        return [], []

    # 2. Remove weak peaks using a relative intensity threshold
    max_intensity = intensities.max()

    if max_intensity <= 0:
        return [], []

    keep = intensities >= min_relative_intensity * max_intensity
    mzs = mzs[keep]
    intensities = intensities[keep]

    if len(mzs) == 0:
        return [], []

    # 3. Scale intensities, apply log transformation, and normalize
    scaled_intensities = 100.0 * intensities / intensities.max()
    transformed_intensities = np.log1p(scaled_intensities)
    transformed_intensities /= transformed_intensities.max()

    # Sort peaks by increasing mass-to-charge ratio
    order = np.argsort(mzs)

    return (
        mzs[order].tolist(),
        transformed_intensities[order].tolist()
    )


# Load only five spectra for a quick test
sample = next(
    pq.ParquetFile(train_path).iter_batches(
        batch_size=5,
        columns=[
            "ms2_mzs",
            "ms2_normalized_intensities",
            "precursor_mz"
        ]
    )
).to_pandas()


for i, row in sample.iterrows():
    cleaned_mzs, cleaned_intensities = clean_spectrum(
        row["ms2_mzs"],
        row["ms2_normalized_intensities"],
        row["precursor_mz"]
    )

    original_mzs = row["ms2_mzs"]
    original_count = 0 if original_mzs is None else len(original_mzs)

    print(f"\nSpectrum {i}")
    print("Number of peaks before cleaning:", original_count)
    print("Number of peaks after cleaning:", len(cleaned_mzs))
    print("First 10 cleaned m/z values:", cleaned_mzs[:10])
    print("First 10 transformed intensities:", cleaned_intensities[:10])




Spectrum 0
Number of peaks before cleaning: 52
Number of peaks after cleaning: 18
First 10 cleaned m/z values: [92.0495, 93.0573, 94.0607, 116.107, 168.0808, 169.0886, 184.0758, 194.06, 209.071, 219.0554]
First 10 transformed intensities: [0.2160480604445317, 0.7628686228404039, 0.15787326498224472, 0.2780375601894059, 0.3488660758295838, 0.3704086026699664, 0.16202938105023537, 0.8845076513323212, 0.2975548249747239, 0.5990087098128838]

Spectrum 1
Number of peaks before cleaning: 28
Number of peaks after cleaning: 17
First 10 cleaned m/z values: [89.0599, 158.9638, 250.0985, 276.0778, 277.0857, 278.0937, 290.0934, 292.1096, 298.0576, 304.1097]
First 10 transformed intensities: [0.991088207233092, 0.8519443031609923, 0.3383696591615161, 0.1654273490528437, 0.3585874886683612, 1.0, 0.16834395840622018, 0.21941479332928643, 0.22281242214201744, 0.4419060570903764]

Spectrum 2
Number of peaks before cleaning: 41
Number of peaks after cleaning: 16
First 10 cleaned m/z values: [111.117, 1

### Clean the train

In [ ]:
import os
import pyarrow as pa
import pyarrow.parquet as pq

train_path = "/content/data/extracted/train.parquet"
output_path = "/content/data/train_cleaned.parquet"

BATCH_SIZE = 10_000

# Remove any incomplete output file from the previous run
if os.path.exists(output_path):
    os.remove(output_path)

parquet_file = pq.ParquetFile(train_path)

# Preserve the original Arrow schema for every batch
original_schema = parquet_file.schema_arrow

writer = None
total_rows = 0
total_peaks_before = 0
total_peaks_after = 0

try:
    for batch_number, batch in enumerate(
        parquet_file.iter_batches(batch_size=BATCH_SIZE),
        start=1
    ):
        df = batch.to_pandas()

        cleaned_mzs_column = []
        cleaned_intensities_column = []

        for _, row in df.iterrows():
            original_mzs = row["ms2_mzs"]
            original_intensities = row["ms2_normalized_intensities"]

            cleaned_mzs, cleaned_intensities = clean_spectrum(
                original_mzs,
                original_intensities,
                row["precursor_mz"]
            )

            cleaned_mzs_column.append(cleaned_mzs)
            cleaned_intensities_column.append(cleaned_intensities)

            total_peaks_before += (
                0 if original_mzs is None else len(original_mzs)
            )
            total_peaks_after += len(cleaned_mzs)

        df["ms2_mzs"] = cleaned_mzs_column
        df["ms2_normalized_intensities"] = cleaned_intensities_column

        # Explicitly enforce the original schema
        table = pa.Table.from_pandas(
            df,
            schema=original_schema,
            preserve_index=False,
            safe=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                original_schema,
                compression="snappy"
            )

        writer.write_table(table)
        total_rows += len(df)

        print(
            f"Batch {batch_number}: "
            f"{total_rows:,} spectra processed"
        )

finally:
    if writer is not None:
        writer.close()

print("\nCleaning completed.")
print(f"Total spectra processed: {total_rows:,}")
print(f"Total peaks before cleaning: {total_peaks_before:,}")
print(f"Total peaks after cleaning: {total_peaks_after:,}")
print(f"Output file: {output_path}")
print(
    f"Output file size: "
    f"{os.path.getsize(output_path) / (1024**3):.2f} GB"
)

Batch 1: 10,000 spectra processed
Batch 2: 20,000 spectra processed
Batch 3: 30,000 spectra processed
Batch 4: 40,000 spectra processed
Batch 5: 50,000 spectra processed
Batch 6: 60,000 spectra processed
Batch 7: 70,000 spectra processed
Batch 8: 80,000 spectra processed
Batch 9: 90,000 spectra processed
Batch 10: 100,000 spectra processed
Batch 11: 110,000 spectra processed
Batch 12: 120,000 spectra processed
Batch 13: 130,000 spectra processed
Batch 14: 140,000 spectra processed
Batch 15: 150,000 spectra processed
Batch 16: 160,000 spectra processed
Batch 17: 170,000 spectra processed
Batch 18: 180,000 spectra processed
Batch 19: 190,000 spectra processed
Batch 20: 200,000 spectra processed
Batch 21: 210,000 spectra processed
Batch 22: 220,000 spectra processed
Batch 23: 230,000 spectra processed
Batch 24: 240,000 spectra processed
Batch 25: 250,000 spectra processed
Batch 26: 260,000 spectra processed
Batch 27: 270,000 spectra processed
Batch 28: 280,000 spectra processed
Batch 29: 

### Clean the test



In [ ]:

import os
import pyarrow as pa
import pyarrow.parquet as pq

test_path = "/content/data/extracted/test.parquet"
output_path = "/content/data/test_cleaned.parquet"

BATCH_SIZE = 1_000

if os.path.exists(output_path):
    os.remove(output_path)

parquet_file = pq.ParquetFile(test_path)
original_schema = parquet_file.schema_arrow

writer = None
total_rows = 0
total_peaks_before = 0
total_peaks_after = 0

try:
    for batch_number, batch in enumerate(
        parquet_file.iter_batches(batch_size=BATCH_SIZE),
        start=1
    ):
        df = batch.to_pandas()

        cleaned_mzs_column = []
        cleaned_intensities_column = []

        for _, row in df.iterrows():
            original_mzs = row["ms2_mzs"]
            original_intensities = row["ms2_normalized_intensities"]

            cleaned_mzs, cleaned_intensities = clean_spectrum(
                original_mzs,
                original_intensities,
                row["precursor_mz"]
            )

            cleaned_mzs_column.append(cleaned_mzs)
            cleaned_intensities_column.append(cleaned_intensities)

            total_peaks_before += (
                0 if original_mzs is None else len(original_mzs)
            )
            total_peaks_after += len(cleaned_mzs)

        df["ms2_mzs"] = cleaned_mzs_column
        df["ms2_normalized_intensities"] = cleaned_intensities_column

        table = pa.Table.from_pandas(
            df,
            schema=original_schema,
            preserve_index=False,
            safe=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                original_schema,
                compression="snappy"
            )

        writer.write_table(table)
        total_rows += len(df)

        print(f"Batch {batch_number}: {total_rows:,} spectra processed")

finally:
    if writer is not None:
        writer.close()

print("\nTest cleaning completed.")
print(f"Total spectra processed: {total_rows:,}")
print(f"Total peaks before cleaning: {total_peaks_before:,}")
print(f"Total peaks after cleaning: {total_peaks_after:,}")
print(f"Output file: {output_path}")
print(f"Output file size: {os.path.getsize(output_path) / (1024**2):.2f} MB")

cleaned_test = pq.read_table(
    output_path,
    columns=["molecule_id", "spectrum_id", "ms2_mzs", "ms2_normalized_intensities"]
).to_pandas()

print("\nOutput validation:")
print("Rows:", len(cleaned_test))
print("Missing molecule IDs:", cleaned_test["molecule_id"].isna().sum())
print("Missing spectrum IDs:", cleaned_test["spectrum_id"].isna().sum())
print(
    "Mass/intensity lengths match:",
    (
        cleaned_test["ms2_mzs"].str.len()
        == cleaned_test["ms2_normalized_intensities"].str.len()
    ).all()
)


Batch 1: 1,000 spectra processed
Batch 2: 1,213 spectra processed

Test cleaning completed.
Total spectra processed: 1,213
Total peaks before cleaning: 400,707
Total peaks after cleaning: 30,331
Output file: /content/data/test_cleaned.parquet
Output file size: 0.50 MB

Output validation:
Rows: 1213
Missing molecule IDs: 0
Missing spectrum IDs: 0
Mass/intensity lengths match: True


### Examine

In [ ]:
import os
import pandas as pd

sample_path = "/content/data/extracted/sample_submission.csv"

if not os.path.exists(sample_path):
    print("Sample submission file not found.")
    print("Available files:")
    for root, dirs, files in os.walk("/content/data"):
        for file in files:
            if "submission" in file.lower():
                print(os.path.join(root, file))
else:
    sample_submission = pd.read_csv(sample_path)

    print("Sample submission shape:", sample_submission.shape)
    print("\nColumns:")
    print(sample_submission.columns.tolist())

    print("\nFirst 5 rows:")
    display(sample_submission.head())

    print("\nData types:")
    print(sample_submission.dtypes)

    print("\nMissing values:")
    print(sample_submission.isna().sum())

Sample submission shape: (400, 2)

Columns:
['molecule_id', 'smiles']

First 5 rows:


,molecule_id,smiles
0,m_005e53,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
1,m_006153,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
2,m_00b5aa,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
3,m_01e922,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
4,m_0259d4,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...



Data types:
molecule_id    object
smiles         object
dtype: object

Missing values:
molecule_id    0
smiles         0
dtype: int64


### Separation of training, Validation, and Metrics

In [ ]:

import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import GroupShuffleSplit

TRAIN_PATH = "/content/data/extracted/train.parquet"
VALIDATION_SIZE = 0.20
RANDOM_STATE = 42

# Load molecule identifiers and labels
metadata = pq.read_table(
    TRAIN_PATH,
    columns=["inchikey14", "normalized_smiles"]
).to_pandas()

# Exclude rows without a valid grouping key
metadata = metadata.dropna(subset=["inchikey14"]).reset_index(drop=True)

# Split by molecular structure, not by individual spectrum
groups = metadata["inchikey14"].astype(str)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(metadata, groups=groups)
)

train_metadata = metadata.iloc[train_idx].copy()
val_metadata = metadata.iloc[val_idx].copy()

# Verify that no molecular structure appears in both sets
train_groups = set(train_metadata["inchikey14"].astype(str))
val_groups = set(val_metadata["inchikey14"].astype(str))

overlap = train_groups.intersection(val_groups)

assert len(overlap) == 0, (
    f"Data leakage detected: {len(overlap)} overlapping molecules."
)

print("Group-based split completed successfully.")
print(f"Training rows: {len(train_metadata):,}")
print(f"Validation rows: {len(val_metadata):,}")
print(f"Training molecules: {len(train_groups):,}")
print(f"Validation molecules: {len(val_groups):,}")
print(f"Overlapping molecules: {len(overlap)}")
print(f"Validation fraction: {len(val_metadata) / len(metadata):.2%}")


Group-based split completed successfully.
Training rows: 2,029,268
Validation rows: 510,340
Training molecules: 220,648
Validation molecules: 55,162
Overlapping molecules: 0
Validation fraction: 20.10%


### MRR@25

In [ ]:

import numpy as np
import pandas as pd


def parse_candidates(prediction):
    """Convert a candidate list or semicolon-separated string into a list."""
    if isinstance(prediction, str):
        candidates = prediction.split(";")
    elif isinstance(prediction, (list, tuple, np.ndarray)):
        candidates = list(prediction)
    else:
        return []

    # Remove empty candidates and preserve ranking order
    result = []
    seen = set()

    for candidate in candidates:
        if not isinstance(candidate, str):
            continue

        candidate = candidate.strip()

        if candidate and candidate not in seen:
            result.append(candidate)
            seen.add(candidate)

    return result


def mrr_at_k(y_true, y_pred, k=25):
    """
    Compute Mean Reciprocal Rank at k.

    y_true: correct SMILES for each molecule.
    y_pred: ranked candidate SMILES for each molecule.
    """
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must have equal lengths.")

    if len(y_true) == 0:
        raise ValueError("Evaluation data must not be empty.")

    reciprocal_ranks = []

    for true_smiles, predictions in zip(y_true, y_pred):
        candidates = parse_candidates(predictions)[:k]
        reciprocal_rank = 0.0

        for rank, candidate in enumerate(candidates, start=1):
            if candidate == true_smiles:
                reciprocal_rank = 1.0 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

    return float(np.mean(reciprocal_ranks))


def hit_rate_at_k(y_true, y_pred, k=25):
    """
    Compute the proportion of molecules with the correct SMILES
    appearing among the first k candidates.
    """
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must have equal lengths.")

    if len(y_true) == 0:
        raise ValueError("Evaluation data must not be empty.")

    hits = []

    for true_smiles, predictions in zip(y_true, y_pred):
        candidates = parse_candidates(predictions)[:k]
        hits.append(true_smiles in candidates)

    return float(np.mean(hits))


def evaluate_predictions(y_true, y_pred):
    """
    Compute MRR@25 and complementary top-k hit rates.
    """
    metrics = {
        "MRR@25": mrr_at_k(y_true, y_pred, k=25),
        "HitRate@1": hit_rate_at_k(y_true, y_pred, k=1),
        "HitRate@5": hit_rate_at_k(y_true, y_pred, k=5),
        "HitRate@10": hit_rate_at_k(y_true, y_pred, k=10),
        "HitRate@25": hit_rate_at_k(y_true, y_pred, k=25),
    }

    return pd.Series(metrics, name="score")


# ============================================================
# EXAMPLE TEST
# ============================================================

y_true_example = [
    "CCO",
    "CCN",
    "CCC",
    "CO"
]

y_pred_example = [
    ["CCO", "CCN", "CCC"],           # Correct at rank 1
    ["CCC", "CCN", "CCO"],           # Correct at rank 2
    ["CCN", "CCO", "CCC"],           # Correct at rank 3
    ["CCC", "CCN", "CCO"]            # Correct candidate absent
]

example_scores = evaluate_predictions(
    y_true_example,
    y_pred_example
)

print("Example evaluation:")
print(example_scores)

# Expected:
# MRR@25     = (1 + 1/2 + 1/3 + 0) / 4 = 0.458333
# HitRate@1  = 0.25
# HitRate@5  = 0.75
# HitRate@10 = 0.75
# HitRate@25 = 0.75


Example evaluation:
MRR@25        0.458333
HitRate@1     0.250000
HitRate@5     0.750000
HitRate@10    0.750000
HitRate@25    0.750000
Name: score, dtype: float64


### Sorting by Mass Proximity

In [ ]:

import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ============================================================
# CONFIGURATION
# ============================================================

TRAIN_PATH = "/content/data/extracted/train.parquet"
TEST_PATH = "/content/data/extracted/test.parquet"
SAMPLE_PATH = "/content/data/extracted/sample_submission.csv"

OUTPUT_PATH = "/content/data/submission_mass_baseline.csv"

TOP_K = 25
LOCAL_WINDOW = 250


# ============================================================
# 1. LOAD THE REQUIRED DATA
# ============================================================

train = pq.read_table(
    TRAIN_PATH,
    columns=[
        "normalized_smiles",
        "inchikey14",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
).to_pandas()

test = pq.read_table(
    TEST_PATH,
    columns=[
        "molecule_id",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
).to_pandas()

sample_submission = pd.read_csv(SAMPLE_PATH)

print("Training rows:", f"{len(train):,}")
print("Test spectra:", f"{len(test):,}")
print("Submission molecules:", f"{len(sample_submission):,}")


# ============================================================
# 2. PREPARE THE TRAINING CANDIDATE INDEX
# ============================================================

train = train.dropna(
    subset=["normalized_smiles", "inchikey14", "precursor_mz"]
).copy()

train["precursor_mz"] = pd.to_numeric(
    train["precursor_mz"], errors="coerce"
)

train = train.dropna(subset=["precursor_mz"])
train = train[train["precursor_mz"] > 0]

# Normalize categorical labels
for column in ["adduct", "ionization_mode"]:
    train[column] = train[column].fillna("UNKNOWN").astype(str).str.strip()

# Keep one representative record per structure and acquisition ion type
candidate_index = (
    train.groupby(
        ["inchikey14", "adduct", "ionization_mode"],
        as_index=False,
        sort=False,
    )
    .agg(
        normalized_smiles=("normalized_smiles", "first"),
        precursor_mz=("precursor_mz", "median"),
    )
)

candidate_index = candidate_index.drop_duplicates(
    subset=["inchikey14", "adduct", "ionization_mode"]
)

print("Unique structure/adduct candidates:", f"{len(candidate_index):,}")


# ============================================================
# 3. BUILD SORTED MASS INDEXES
# ============================================================

def make_mass_pool(frame):
    """Sort a candidate pool by precursor mass."""
    frame = frame.sort_values("precursor_mz").reset_index(drop=True)

    return {
        "frame": frame,
        "masses": frame["precursor_mz"].to_numpy(dtype=float),
    }


pools_by_adduct_mode = {}
pools_by_adduct = {}

for key, group in candidate_index.groupby(
    ["adduct", "ionization_mode"], sort=False
):
    pools_by_adduct_mode[key] = make_mass_pool(group)

for key, group in candidate_index.groupby("adduct", sort=False):
    pools_by_adduct[key] = make_mass_pool(group)

global_pool = make_mass_pool(candidate_index)


# ============================================================
# 4. RANK CANDIDATES BY PRECURSOR MASS ERROR
# ============================================================

def normalize_label(value):
    if pd.isna(value):
        return "UNKNOWN"
    return str(value).strip()


def retrieve_mass_candidates(
    precursor_mz,
    adduct,
    ionization_mode,
    top_k=TOP_K,
):
    """
    Retrieve candidates with the closest precursor masses.
    Candidates are prioritized by matching adduct and ion mode.
    """
    if pd.isna(precursor_mz):
        return []

    precursor_mz = float(precursor_mz)

    if not np.isfinite(precursor_mz) or precursor_mz <= 0:
        return []

    adduct = normalize_label(adduct)
    ionization_mode = normalize_label(ionization_mode)

    pool = pools_by_adduct_mode.get((adduct, ionization_mode))

    if pool is None or len(pool["masses"]) == 0:
        pool = pools_by_adduct.get(adduct)

    if pool is None or len(pool["masses"]) == 0:
        pool = global_pool

    masses = pool["masses"]
    frame = pool["frame"]

    insertion = int(np.searchsorted(masses, precursor_mz))

    # Restrict the search to a local window around the target mass
    start = max(0, insertion - LOCAL_WINDOW)
    end = min(len(masses), insertion + LOCAL_WINDOW)

    local_frame = frame.iloc[start:end].copy()

    if local_frame.empty:
        return []

    local_frame["mass_error_ppm"] = (
        np.abs(local_frame["precursor_mz"] - precursor_mz)
        / precursor_mz
        * 1e6
    )

    local_frame = local_frame.sort_values(
        "mass_error_ppm"
    ).drop_duplicates("inchikey14")

    return local_frame[
        ["inchikey14", "normalized_smiles", "mass_error_ppm"]
    ].head(top_k).to_dict("records")


# ============================================================
# 5. AGGREGATE CANDIDATES AT MOLECULE LEVEL
# ============================================================

spectra_by_molecule = {
    molecule_id: group
    for molecule_id, group in test.groupby("molecule_id", sort=False)
}

# Frequent structures provide a fallback if a molecule has no usable spectrum
fallback_smiles = (
    train["normalized_smiles"]
    .value_counts()
    .head(TOP_K)
    .index
    .tolist()
)

submission_rows = []
molecules_without_candidates = 0

for molecule_id in sample_submission["molecule_id"]:
    spectra = spectra_by_molecule.get(molecule_id)

    # Structure ID -> best mass error observed across its spectra
    molecule_scores = {}

    if spectra is not None:
        for row in spectra.itertuples(index=False):
            candidates = retrieve_mass_candidates(
                precursor_mz=row.precursor_mz,
                adduct=row.adduct,
                ionization_mode=row.ionization_mode,
                top_k=TOP_K,
            )

            for candidate in candidates:
                structure_id = candidate["inchikey14"]
                smiles = candidate["normalized_smiles"]
                error = candidate["mass_error_ppm"]

                previous = molecule_scores.get(structure_id)

                if previous is None or error < previous[0]:
                    molecule_scores[structure_id] = (error, smiles)

    # Rank candidates by their best observed precursor-mass error
    ranked = sorted(
        molecule_scores.values(),
        key=lambda item: item[0],
    )

    ranked_smiles = []

    for _, smiles in ranked:
        if smiles not in ranked_smiles:
            ranked_smiles.append(smiles)

        if len(ranked_smiles) >= TOP_K:
            break

    if not ranked_smiles:
        molecules_without_candidates += 1
        ranked_smiles = fallback_smiles.copy()

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(ranked_smiles[:TOP_K]),
    })


# ============================================================
# 6. SAVE AND VALIDATE THE SUBMISSION
# ============================================================

submission = pd.DataFrame(submission_rows)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert submission["molecule_id"].equals(
    sample_submission["molecule_id"]
)
assert submission["smiles"].notna().all()

candidate_counts = submission["smiles"].str.split(";").str.len()

print("\nBASELINE RESULTS")
print("-" * 45)
print("Submission rows:", len(submission))
print("Molecules without retrieved candidates:", molecules_without_candidates)
print("Minimum candidates per molecule:", candidate_counts.min())
print("Maximum candidates per molecule:", candidate_counts.max())
print("Mean candidates per molecule:", round(candidate_counts.mean(), 2))
print("Output file:", OUTPUT_PATH)

print("\nFirst 5 predictions:")
print(submission.head().to_string(index=False))


Training rows: 2,539,608
Test spectra: 1,213
Submission molecules: 400
Unique structure/adduct candidates: 521,361

BASELINE RESULTS
---------------------------------------------
Submission rows: 400
Molecules without retrieved candidates: 0
Minimum candidates per molecule: 25
Maximum candidates per molecule: 25
Mean candidates per molecule: 25.0
Output file: /content/data/submission_mass_baseline.csv

First 5 predictions:
molecule_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

### Similarity


In [ ]:

import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from collections import defaultdict


# ============================================================
# CONFIGURATION
# ============================================================

TRAIN_PATH = "/content/data/extracted/train.parquet"
TEST_PATH = "/content/data/extracted/test.parquet"
SAMPLE_PATH = "/content/data/extracted/sample_submission.csv"

OUTPUT_PATH = "/content/data/submission_cosine_baseline.csv"

BATCH_SIZE = 50_000
CANDIDATES_PER_SPECTRUM = 100
FRAGMENT_BIN_WIDTH = 0.1
PRECURSOR_TOLERANCE_PPM = 1000
MIN_ABSOLUTE_TOLERANCE = 0.5
PRECURSOR_PEAK_TOLERANCE_PPM = 10


# ============================================================
# 1. LOAD TEST DATA AND TRAINING METADATA
# ============================================================

test = pq.read_table(
    TEST_PATH,
    columns=[
        "molecule_id",
        "spectrum_id",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "ms2_mzs",
        "ms2_normalized_intensities",
    ],
).to_pandas()

sample_submission = pd.read_csv(SAMPLE_PATH)

parquet_file = pq.ParquetFile(TRAIN_PATH)

metadata_parts = []
row_offset = 0

for batch in parquet_file.iter_batches(
    batch_size=BATCH_SIZE,
    columns=[
        "normalized_smiles",
        "inchikey14",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
):
    part = batch.to_pandas()
    part["_row_id"] = np.arange(
        row_offset, row_offset + len(part), dtype=np.int64
    )
    row_offset += len(part)
    metadata_parts.append(part)

train_meta = pd.concat(metadata_parts, ignore_index=True)
del metadata_parts

train_meta["precursor_mz"] = pd.to_numeric(
    train_meta["precursor_mz"], errors="coerce"
)

train_meta = train_meta.dropna(
    subset=["normalized_smiles", "inchikey14", "precursor_mz"]
).copy()

train_meta = train_meta[
    np.isfinite(train_meta["precursor_mz"])
    & (train_meta["precursor_mz"] > 0)
].copy()

for column in ["adduct", "ionization_mode"]:
    train_meta[column] = (
        train_meta[column]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )

print("Training reference spectra:", f"{len(train_meta):,}")
print("Test spectra:", f"{len(test):,}")
print("Test molecules:", test["molecule_id"].nunique())


# ============================================================
# 2. BUILD MASS-SORTED REFERENCE POOLS
# ============================================================

def make_pool(frame):
    frame = frame.sort_values("precursor_mz").reset_index(drop=True)

    return {
        "frame": frame,
        "masses": frame["precursor_mz"].to_numpy(dtype=np.float64),
        "row_ids": frame["_row_id"].to_numpy(dtype=np.int64),
    }


pools_by_adduct_mode = {}
pools_by_adduct = {}

for key, group in train_meta.groupby(
    ["adduct", "ionization_mode"], sort=False
):
    pools_by_adduct_mode[key] = make_pool(group)

for key, group in train_meta.groupby("adduct", sort=False):
    pools_by_adduct[key] = make_pool(group)

global_pool = make_pool(train_meta)

# Fast lookup of candidate metadata by original parquet row ID
meta_by_row = train_meta.set_index("_row_id")


# ============================================================
# 3. RETRIEVE REFERENCES NEAR THE PRECURSOR MASS
# ============================================================

def normalize_label(value):
    if pd.isna(value):
        return "UNKNOWN"
    return str(value).strip()


def retrieve_references(precursor_mz, adduct, ion_mode):
    if pd.isna(precursor_mz):
        return []

    precursor_mz = float(precursor_mz)

    if not np.isfinite(precursor_mz) or precursor_mz <= 0:
        return []

    adduct = normalize_label(adduct)
    ion_mode = normalize_label(ion_mode)

    pool = pools_by_adduct_mode.get((adduct, ion_mode))

    if pool is None:
        pool = pools_by_adduct.get(adduct)

    if pool is None:
        pool = global_pool

    masses = pool["masses"]
    row_ids = pool["row_ids"]

    insertion = np.searchsorted(masses, precursor_mz)

    tolerance = max(
        MIN_ABSOLUTE_TOLERANCE,
        precursor_mz * PRECURSOR_TOLERANCE_PPM * 1e-6,
    )

    left = np.searchsorted(masses, precursor_mz - tolerance)
    right = np.searchsorted(masses, precursor_mz + tolerance)

    candidate_masses = masses[left:right]
    candidate_rows = row_ids[left:right]

    # If no references fall inside the mass window, use the nearest
    # references from the selected ion-type pool as a fallback.
    if len(candidate_rows) == 0:
        start = max(0, insertion - CANDIDATES_PER_SPECTRUM)
        end = min(len(masses), insertion + CANDIDATES_PER_SPECTRUM)

        candidate_masses = masses[start:end]
        candidate_rows = row_ids[start:end]

    if len(candidate_rows) == 0:
        return []

    errors_ppm = (
        np.abs(candidate_masses - precursor_mz)
        / precursor_mz
        * 1e6
    )

    order = np.argsort(errors_ppm)[:CANDIDATES_PER_SPECTRUM]

    return [
        (int(candidate_rows[i]), float(errors_ppm[i]))
        for i in order
    ]


# Candidate references for each test spectrum
references_by_spectrum = []

for row in test.itertuples(index=False):
    references_by_spectrum.append(
        retrieve_references(
            row.precursor_mz,
            row.adduct,
            row.ionization_mode,
        )
    )

selected_row_ids = sorted({
    row_id
    for references in references_by_spectrum
    for row_id, _ in references
})

print("Unique reference spectra selected:", f"{len(selected_row_ids):,}")


# ============================================================
# 4. LOAD ONLY SELECTED REFERENCE SPECTRA
# ============================================================

selected_row_ids = np.asarray(selected_row_ids, dtype=np.int64)
selected_reference_set = set(selected_row_ids.tolist())

reference_spectra = {}
row_offset = 0

for batch in parquet_file.iter_batches(
    batch_size=BATCH_SIZE,
    columns=["ms2_mzs", "ms2_normalized_intensities"],
):
    batch_size = batch.num_rows
    batch_end = row_offset + batch_size

    start_pos = np.searchsorted(selected_row_ids, row_offset)
    end_pos = np.searchsorted(selected_row_ids, batch_end)

    if start_pos < end_pos:
        batch_row_ids = selected_row_ids[start_pos:end_pos]
        local_positions = batch_row_ids - row_offset

        mz_lists = batch.column("ms2_mzs").to_pylist()
        intensity_lists = batch.column(
            "ms2_normalized_intensities"
        ).to_pylist()

        for row_id, local_pos in zip(batch_row_ids, local_positions):
            reference_spectra[int(row_id)] = (
                mz_lists[int(local_pos)],
                intensity_lists[int(local_pos)],
            )

    row_offset = batch_end

print("Reference spectra loaded:", f"{len(reference_spectra):,}")


# ============================================================
# 5. CONVERT A SPECTRUM TO A NORMALIZED BINNED VECTOR
# ============================================================

def spectrum_to_vector(
    mzs,
    intensities,
    precursor_mz=None,
):
    """
    Bin fragment masses and combine intensities within each bin.
    Returns a sparse dictionary: bin_index -> normalized intensity.
    """
    if mzs is None or intensities is None:
        return {}

    mzs = np.asarray(mzs, dtype=np.float64)
    intensities = np.asarray(intensities, dtype=np.float64)

    n = min(len(mzs), len(intensities))
    mzs = mzs[:n]
    intensities = intensities[:n]

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities > 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    if len(mzs) == 0:
        return {}

    # Remove the precursor peak if it is present in the fragment list
    if precursor_mz is not None and pd.notna(precursor_mz):
        precursor_mz = float(precursor_mz)

        if np.isfinite(precursor_mz) and precursor_mz > 0:
            tolerance = max(
                0.005,
                precursor_mz * PRECURSOR_PEAK_TOLERANCE_PPM * 1e-6,
            )

            keep = np.abs(mzs - precursor_mz) > tolerance
            mzs = mzs[keep]
            intensities = intensities[keep]

    if len(mzs) == 0:
        return {}

    # Remove very weak peaks relative to the spectrum's base peak
    max_intensity = intensities.max()

    keep = intensities >= 0.005 * max_intensity
    mzs = mzs[keep]
    intensities = intensities[keep]

    if len(mzs) == 0:
        return {}

    # Relative intensities, followed by square-root scaling
    intensities = np.sqrt(intensities / intensities.max())

    bins = np.rint(mzs / FRAGMENT_BIN_WIDTH).astype(np.int64)

    # Merge intensities that fall into the same fragment-mass bin
    unique_bins, inverse = np.unique(bins, return_inverse=True)

    merged = np.zeros(len(unique_bins), dtype=np.float64)
    np.add.at(merged, inverse, intensities)

    norm = np.linalg.norm(merged)

    if norm == 0:
        return {}

    merged /= norm

    return dict(zip(unique_bins.tolist(), merged.tolist()))


def cosine_similarity_sparse(vector_a, vector_b):
    """Cosine similarity between two sparse, normalized dictionaries."""
    if not vector_a or not vector_b:
        return 0.0

    if len(vector_a) > len(vector_b):
        vector_a, vector_b = vector_b, vector_a

    return float(sum(
        value * vector_b.get(bin_id, 0.0)
        for bin_id, value in vector_a.items()
    ))


# ============================================================
# 6. SCORE STRUCTURES BY SPECTRAL COSINE SIMILARITY
# ============================================================

molecule_scores = defaultdict(dict)
scored_spectra = 0
spectra_without_valid_vector = 0

for spectrum_position, row in enumerate(test.itertuples(index=False)):
    query_vector = spectrum_to_vector(
        row.ms2_mzs,
        row.ms2_normalized_intensities,
        row.precursor_mz,
    )

    if not query_vector:
        spectra_without_valid_vector += 1
        continue

    references = references_by_spectrum[spectrum_position]
    best_by_structure = {}

    for reference_row_id, mass_error_ppm in references:
        reference = reference_spectra.get(reference_row_id)

        if reference is None:
            continue

        ref_mzs, ref_intensities = reference

        ref_meta = meta_by_row.loc[reference_row_id]

        ref_vector = spectrum_to_vector(
            ref_mzs,
            ref_intensities,
            ref_meta["precursor_mz"],
        )

        score = cosine_similarity_sparse(query_vector, ref_vector)

        structure_id = str(ref_meta["inchikey14"])
        smiles = str(ref_meta["normalized_smiles"])

        previous = best_by_structure.get(structure_id)

        if (
            previous is None
            or score > previous[0]
            or (
                score == previous[0]
                and mass_error_ppm < previous[1]
            )
        ):
            best_by_structure[structure_id] = (
                score,
                mass_error_ppm,
                smiles,
            )

    # Aggregate across all spectra belonging to the same test molecule
    molecule_dict = molecule_scores[row.molecule_id]

    for structure_id, (score, mass_error_ppm, smiles) in best_by_structure.items():
        previous = molecule_dict.get(structure_id)

        if (
            previous is None
            or score > previous[0]
            or (
                score == previous[0]
                and mass_error_ppm < previous[1]
            )
        ):
            molecule_dict[structure_id] = (
                score,
                mass_error_ppm,
                smiles,
            )

    scored_spectra += 1

    if scored_spectra % 100 == 0:
        print(f"Scored test spectra: {scored_spectra:,}")


# ============================================================
# 7. CREATE THE RANKED SUBMISSION
# ============================================================

fallback_smiles = (
    train_meta["normalized_smiles"]
    .value_counts()
    .head(25)
    .index
    .tolist()
)

submission_rows = []
molecules_without_candidates = 0

for molecule_id in sample_submission["molecule_id"]:
    structure_scores = molecule_scores.get(molecule_id, {})

    ranked_structures = sorted(
        structure_scores.values(),
        key=lambda item: (-item[0], item[1]),
    )

    ranked_smiles = []

    for cosine_score, mass_error_ppm, smiles in ranked_structures:
        if smiles not in ranked_smiles:
            ranked_smiles.append(smiles)

        if len(ranked_smiles) == 25:
            break

    if not ranked_smiles:
        molecules_without_candidates += 1
        ranked_smiles = fallback_smiles.copy()

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(ranked_smiles[:25]),
    })


submission = pd.DataFrame(submission_rows)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert submission["molecule_id"].equals(
    sample_submission["molecule_id"]
)
assert submission["smiles"].notna().all()

print("\nCOSINE BASELINE SUMMARY")
print("-" * 50)
print("Test spectra scored:", scored_spectra)
print("Spectra without usable fragments:", spectra_without_valid_vector)
print("Molecules without retrieved candidates:", molecules_without_candidates)
print("Submission rows:", len(submission))
print("Output file:", OUTPUT_PATH)

print("\nFirst 5 predictions:")
print(submission.head().to_string(index=False))


Training reference spectra: 2,539,608
Test spectra: 1,213
Test molecules: 400
Unique reference spectra selected: 51,032
Reference spectra loaded: 51,032
Scored test spectra: 100
Scored test spectra: 200
Scored test spectra: 300
Scored test spectra: 400
Scored test spectra: 500
Scored test spectra: 600
Scored test spectra: 700
Scored test spectra: 800
Scored test spectra: 900
Scored test spectra: 1,000
Scored test spectra: 1,100
Scored test spectra: 1,200

COSINE BASELINE SUMMARY
--------------------------------------------------
Test spectra scored: 1213
Spectra without usable fragments: 0
Molecules without retrieved candidates: 0
Submission rows: 400
Output file: /content/data/submission_cosine_baseline.csv

First 5 predictions:
molecule_id                                                                                                                                                                                                                                                         

### Start the local assessment

In [ ]:

# ============================================================
# COMPLETE LOCAL EVALUATION
# Mass-based ranking vs MS/MS cosine similarity
# Competition: Enveda CASMI 2026
# ============================================================

import os
import gc
import warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from scipy.sparse import coo_matrix
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

TRAIN_PATH = "/content/data/extracted/train.parquet"
OUTPUT_DIR = "/content/data"

SEED = 42
MAX_VALIDATION_MOLECULES = 10_000
CANDIDATE_POOL_SIZE = 50
TOP_K = 25

BIN_WIDTH = 0.1
MIN_RELATIVE_INTENSITY = 0.005
PRECURSOR_TOLERANCE_PPM = 10.0

# Small batches reduce peak RAM usage in Colab.
BATCH_SIZE = 10_000

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"File not found: {TRAIN_PATH}")

print("Configuration loaded.")
print("Training file:", TRAIN_PATH)

# ------------------------------------------------------------
# 2. READ METADATA IN BATCHES
# ------------------------------------------------------------

print("\n[1/7] Reading metadata...")

parquet_file = pq.ParquetFile(TRAIN_PATH)

required_columns = [
    "normalized_smiles",
    "inchikey14",
    "precursor_mz",
    "adduct",
    "ionization_mode",
    "ms2_mzs",
    "ms2_normalized_intensities",
]

available_columns = parquet_file.schema_arrow.names
missing = [c for c in required_columns if c not in available_columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

metadata_parts = []
row_offset = 0

for batch in parquet_file.iter_batches(
    batch_size=100_000,
    columns=[
        "normalized_smiles",
        "inchikey14",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
):
    part = batch.to_pandas()
    part["row_id"] = np.arange(
        row_offset,
        row_offset + len(part),
        dtype=np.int64,
    )
    row_offset += len(part)
    metadata_parts.append(part)

    if row_offset % 500_000 < 100_000:
        print(f"  Read {row_offset:,} rows")

total_rows = row_offset
metadata = pd.concat(metadata_parts, ignore_index=True)
del metadata_parts
gc.collect()

metadata["precursor_mz"] = pd.to_numeric(
    metadata["precursor_mz"], errors="coerce"
)

metadata = metadata.dropna(
    subset=["normalized_smiles", "inchikey14", "precursor_mz"]
).copy()

metadata = metadata[
    (metadata["precursor_mz"] > 0)
    & (metadata["inchikey14"].astype(str).str.len() > 0)
    & (metadata["normalized_smiles"].astype(str).str.len() > 0)
].copy()

metadata["inchikey14"] = metadata["inchikey14"].astype(str)

def clean_key(value):
    if pd.isna(value):
        return "__MISSING__"
    return str(value)

metadata["adduct_key"] = metadata["adduct"].map(clean_key)
metadata["mode_key"] = metadata["ionization_mode"].map(clean_key)

print(f"Usable rows: {len(metadata):,}")
print(f"Unique structures: {metadata['inchikey14'].nunique():,}")

# ------------------------------------------------------------
# 3. SPECTRUM-LEVEL VALIDATION SPLIT
# ------------------------------------------------------------

print("\n[2/7] Creating a spectrum-level split...")

# We require at least two spectra of the same structure
# with the same adduct and ionization mode.
# One spectrum can then be held out while another remains
# available as a reference.

metadata["group_count"] = metadata.groupby(
    ["inchikey14", "adduct_key", "mode_key"],
    sort=False,
)["row_id"].transform("size")

eligible = metadata[metadata["group_count"] >= 2].copy()

if eligible.empty:
    raise RuntimeError(
        "No structures have two spectra with the same adduct/mode."
    )

# Select one eligible spectrum per structure.
validation_meta = (
    eligible.sample(frac=1, random_state=SEED)
    .drop_duplicates("inchikey14")
    .copy()
)

if len(validation_meta) > MAX_VALIDATION_MOLECULES:
    validation_meta = validation_meta.sample(
        n=MAX_VALIDATION_MOLECULES,
        random_state=SEED,
    )

validation_meta = validation_meta.sort_values(
    "row_id"
).reset_index(drop=True)

validation_ids = set(
    validation_meta["row_id"].astype(int)
)

# Keep all other spectra, including other spectra of
# validation structures, in the reference library.
train_meta = metadata[
    ~metadata["row_id"].isin(validation_ids)
].copy()

reference_structures = set(train_meta["inchikey14"])
validation_structures = set(validation_meta["inchikey14"])

missing_truths = validation_structures - reference_structures

print(f"Reference spectra: {len(train_meta):,}")
print(f"Validation spectra: {len(validation_meta):,}")
print(f"Validation structures: {len(validation_structures):,}")
print(f"Validation structures absent from reference: {len(missing_truths)}")

if missing_truths:
    raise RuntimeError(
        "Some validation structures have no reference spectrum."
    )

# Confirm each validation structure has a reference in the same
# adduct/mode group, so the candidate search is meaningful.
reference_groups = set(
    zip(
        train_meta["inchikey14"],
        train_meta["adduct_key"],
        train_meta["mode_key"],
    )
)

same_group_available = sum(
    (
        row.inchikey14,
        row.adduct_key,
        row.mode_key,
    ) in reference_groups
    for row in validation_meta.itertuples(index=False)
)

print(
    "Validation structures with matching adduct/mode reference:",
    f"{same_group_available:,}/{len(validation_meta):,}",
)

if same_group_available != len(validation_meta):
    raise RuntimeError(
        "Some validation spectra lack a same-structure reference "
        "with matching adduct/mode."
    )

# Save arrays indexed by the original Parquet row ID.
query_ids = validation_meta["row_id"].to_numpy(dtype=np.int64)
query_masses = validation_meta["precursor_mz"].to_numpy(dtype=np.float64)
query_truths = validation_meta["inchikey14"].astype(str).to_numpy()

# ------------------------------------------------------------
# 4. BUILD REFERENCE POOLS AND CANDIDATE LISTS
# ------------------------------------------------------------

print("\n[3/7] Building mass-sorted reference pools...")

reference_pools = {}

for key, group in train_meta.groupby(
    ["adduct_key", "mode_key"], sort=False
):
    group = group.sort_values("precursor_mz")

    reference_pools[key] = (
        group["precursor_mz"].to_numpy(dtype=np.float64),
        group["row_id"].to_numpy(dtype=np.int64),
        group["inchikey14"].astype(str).to_numpy(),
    )

print(f"Adduct/mode pools: {len(reference_pools):,}")

def get_candidates(query_mass, pool, limit=50):
    """Retrieve distinct structures nearest in precursor mass."""
    masses, row_ids, structures = pool

    if len(masses) == 0 or not np.isfinite(query_mass):
        return []

    pos = int(np.searchsorted(masses, query_mass))
    left = pos - 1
    right = pos

    candidates = []
    seen = set()

    while len(candidates) < limit and (
        left >= 0 or right < len(masses)
    ):
        if left < 0:
            idx = right
            right += 1
        elif right >= len(masses):
            idx = left
            left -= 1
        elif abs(masses[left] - query_mass) <= abs(
            masses[right] - query_mass
        ):
            idx = left
            left -= 1
        else:
            idx = right
            right += 1

        structure = str(structures[idx])

        if structure in seen:
            continue

        seen.add(structure)

        ppm_error = (
            abs(float(masses[idx]) - query_mass)
            / query_mass
            * 1_000_000
        )

        candidates.append(
            (int(row_ids[idx]), structure, float(ppm_error))
        )

    return candidates


candidate_lists = []

for i, row in enumerate(validation_meta.itertuples(index=False)):
    key = (row.adduct_key, row.mode_key)
    pool = reference_pools.get(key)

    if pool is None:
        candidate_lists.append([])
        continue

    candidates = get_candidates(
        float(row.precursor_mz),
        pool,
        limit=CANDIDATE_POOL_SIZE,
    )
    candidate_lists.append(candidates)

    if (i + 1) % 2000 == 0:
        print(f"  Candidate lists: {i + 1:,}/{len(validation_meta):,}")

# Candidate recall is a crucial diagnostic.
candidate_recall = np.mean([
    any(c[1] == truth for c in candidates)
    for truth, candidates in zip(query_truths, candidate_lists)
])

print(f"Candidate recall@{CANDIDATE_POOL_SIZE}: {candidate_recall:.6f}")

if candidate_recall == 0:
    raise RuntimeError(
        "Candidate recall is zero. Do not continue to cosine scoring: "
        "the candidate generator is not retrieving the known structures."
    )

selected_reference_ids = {
    candidate[0]
    for candidates in candidate_lists
    for candidate in candidates
}

selected_ids = np.array(
    sorted(selected_reference_ids.union(set(map(int, query_ids)))),
    dtype=np.int64,
)

selected_id_to_local = {
    int(row_id): i for i, row_id in enumerate(selected_ids)
}

print(f"Reference spectra selected: {len(selected_reference_ids):,}")
print(f"Unique spectra needed: {len(selected_ids):,}")

# ------------------------------------------------------------
# 5. LOAD SELECTED SPECTRA AS SPARSE VECTORS
# ------------------------------------------------------------

print("\n[4/7] Loading selected MS/MS spectra...")

def spectrum_to_bins(mzs, intensities, precursor_mz):
    if mzs is None or intensities is None:
        return None, None

    try:
        mzs = np.asarray(mzs, dtype=np.float64).ravel()
        intensities = np.asarray(intensities, dtype=np.float64).ravel()
    except (TypeError, ValueError):
        return None, None

    if len(mzs) == 0 or len(mzs) != len(intensities):
        return None, None

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities > 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    if len(mzs) == 0:
        return None, None

    if np.isfinite(precursor_mz) and precursor_mz > 0:
        ppm = (
            np.abs(mzs - precursor_mz)
            / precursor_mz
            * 1_000_000
        )
        keep = ppm > PRECURSOR_TOLERANCE_PPM
        mzs = mzs[keep]
        intensities = intensities[keep]

    if len(mzs) == 0:
        return None, None

    relative = intensities / intensities.max()
    keep = relative >= MIN_RELATIVE_INTENSITY

    mzs = mzs[keep]
    relative = relative[keep]

    if len(mzs) == 0:
        return None, None

    bins = np.floor(mzs / BIN_WIDTH).astype(np.int32)
    values = np.sqrt(relative)

    unique_bins, inverse = np.unique(bins, return_inverse=True)
    combined = np.bincount(
        inverse, weights=values
    ).astype(np.float32)

    return unique_bins, combined


coo_rows = []
coo_cols = []
coo_values = []
loaded_ids = set()
row_offset = 0

for batch in parquet_file.iter_batches(
    batch_size=BATCH_SIZE,
    columns=[
        "precursor_mz",
        "ms2_mzs",
        "ms2_normalized_intensities",
    ],
):
    batch_len = batch.num_rows
    batch_end = row_offset + batch_len

    lo = np.searchsorted(selected_ids, row_offset)
    hi = np.searchsorted(selected_ids, batch_end)
    wanted_ids = selected_ids[lo:hi]

    if len(wanted_ids):
        local_positions = wanted_ids - row_offset

        precursor_values = batch.column("precursor_mz").to_pylist()
        mz_values = batch.column("ms2_mzs").to_pylist()
        intensity_values = batch.column(
            "ms2_normalized_intensities"
        ).to_pylist()

        for row_id, local_pos in zip(wanted_ids, local_positions):
            row_id = int(row_id)
            local_pos = int(local_pos)

            if row_id in loaded_ids:
                continue

            bins, values = spectrum_to_bins(
                mz_values[local_pos],
                intensity_values[local_pos],
                precursor_values[local_pos],
            )

            loaded_ids.add(row_id)

            if bins is None:
                continue

            local_row = selected_id_to_local[row_id]

            coo_rows.append(
                np.full(len(bins), local_row, dtype=np.int32)
            )
            coo_cols.append(bins)
            coo_values.append(values)

    row_offset = batch_end

    if row_offset % 500_000 < BATCH_SIZE:
        print(
            f"  Scanned {row_offset:,}/{total_rows:,}; "
            f"loaded {len(loaded_ids):,} selected spectra"
        )

if not coo_rows:
    raise RuntimeError("No usable MS/MS spectra were loaded.")

all_rows = np.concatenate(coo_rows)
all_cols = np.concatenate(coo_cols)
all_values = np.concatenate(coo_values)

n_bins = int(all_cols.max()) + 1

spectra_matrix = coo_matrix(
    (all_values, (all_rows, all_cols)),
    shape=(len(selected_ids), n_bins),
    dtype=np.float32,
).tocsr()

spectra_matrix = normalize(
    spectra_matrix,
    norm="l2",
    axis=1,
    copy=False,
).tocsr()

del coo_rows, coo_cols, coo_values
del all_rows, all_cols, all_values
gc.collect()

print(f"Sparse matrix shape: {spectra_matrix.shape}")
print(f"Non-zero entries: {spectra_matrix.nnz:,}")

def get_vector(row_id):
    local = selected_id_to_local.get(int(row_id))
    if local is None:
        return None
    return spectra_matrix[local]


# ------------------------------------------------------------
# 6. RANK CANDIDATES WITH MASS AND COSINE SIMILARITY
# ------------------------------------------------------------

print("\n[5/7] Ranking candidates...")

evaluation_results = []

for i, candidates in enumerate(candidate_lists):
    truth = str(query_truths[i])
    query_vector = get_vector(query_ids[i])

    # Method A: mass proximity.
    mass_ranked = [
        c[1] for c in sorted(candidates, key=lambda c: c[2])
    ]

    # Method B: cosine similarity on the same candidate pool.
    if query_vector is None or query_vector.nnz == 0 or not candidates:
        cosine_ranked = mass_ranked.copy()
    else:
        valid_candidates = []
        candidate_local_rows = []

        for candidate in candidates:
            local = selected_id_to_local.get(candidate[0])
            if local is not None:
                valid_candidates.append(candidate)
                candidate_local_rows.append(local)

        if candidate_local_rows:
            reference_vectors = spectra_matrix[candidate_local_rows]
            scores = (
                reference_vectors @ query_vector.T
            ).toarray().ravel()

            order = sorted(
                range(len(valid_candidates)),
                key=lambda j: (
                    -float(scores[j]),
                    valid_candidates[j][2],
                ),
            )

            cosine_ranked = [
                valid_candidates[j][1] for j in order
            ]
        else:
            cosine_ranked = mass_ranked.copy()

    evaluation_results.append({
        "true_structure": truth,
        "mass_ranked": mass_ranked,
        "cosine_ranked": cosine_ranked,
        "candidate_count": len(candidates),
        "truth_in_candidates": truth in mass_ranked,
    })

    if (i + 1) % 2000 == 0:
        print(f"  Evaluated {i + 1:,}/{len(candidate_lists):,}")

# ------------------------------------------------------------
# 7. METRICS AND OUTPUT FILES
# ------------------------------------------------------------

print("\n[6/7] Computing metrics...")

def reciprocal_rank(ranking, truth, k=25):
    for rank, structure in enumerate(ranking[:k], start=1):
        if structure == truth:
            return 1.0 / rank
    return 0.0

def hit_rate(ranking, truth, k):
    return float(truth in ranking[:k])

def compute_metrics(results, method):
    mrr = []
    hits = {k: [] for k in [1, 5, 10, 25]}

    for result in results:
        ranking = result[method]
        truth = result["true_structure"]

        mrr.append(reciprocal_rank(ranking, truth, TOP_K))

        for k in hits:
            hits[k].append(hit_rate(ranking, truth, k))

    output = {"MRR@25": float(np.mean(mrr))}

    for k in hits:
        output[f"HitRate@{k}"] = float(np.mean(hits[k]))

    return output


mass_metrics = compute_metrics(evaluation_results, "mass_ranked")
cosine_metrics = compute_metrics(evaluation_results, "cosine_ranked")

comparison = pd.DataFrame({
    "Mass only": mass_metrics,
    "Cosine similarity": cosine_metrics,
})

comparison["Difference"] = (
    comparison["Cosine similarity"] - comparison["Mass only"]
)

print("\n" + "=" * 70)
print("LOCAL VALIDATION RESULTS")
print("=" * 70)
print(comparison.round(6).to_string())
print("=" * 70)

truth_recall = np.mean([
    result["truth_in_candidates"]
    for result in evaluation_results
])

print(f"\nCandidate recall: {truth_recall:.6f}")
print(f"Validation molecules: {len(evaluation_results):,}")

summary_path = os.path.join(
    OUTPUT_DIR, "local_evaluation_results.csv"
)
comparison.to_csv(summary_path)

rankings_path = os.path.join(
    OUTPUT_DIR, "local_evaluation_rankings.csv"
)

pd.DataFrame([
    {
        "true_inchikey14": result["true_structure"],
        "candidate_count": result["candidate_count"],
        "truth_in_candidates": result["truth_in_candidates"],
        "mass_top25": ";".join(result["mass_ranked"][:25]),
        "cosine_top25": ";".join(result["cosine_ranked"][:25]),
    }
    for result in evaluation_results
]).to_csv(rankings_path, index=False)

print("\n[7/7] Finished.")
print("Metrics:", summary_path)
print("Rankings:", rankings_path)

if truth_recall < 1.0:
    print(
        "\nWARNING: Some correct structures were not retrieved "
        "in the candidate pool. MRR and HitRate are affected by "
        "candidate generation as well as ranking."
    )


Configuration loaded.
Training file: /content/data/extracted/train.parquet

[1/7] Reading metadata...
  Read 500,000 rows
  Read 1,000,000 rows
  Read 1,500,000 rows
  Read 2,000,000 rows
  Read 2,500,000 rows
  Read 2,539,608 rows
Usable rows: 2,539,608
Unique structures: 275,810

[2/7] Creating a spectrum-level split...
Reference spectra: 2,529,608
Validation spectra: 10,000
Validation structures: 10,000
Validation structures absent from reference: 0
Validation structures with matching adduct/mode reference: 10,000/10,000

[3/7] Building mass-sorted reference pools...
Adduct/mode pools: 132
  Candidate lists: 2,000/10,000
  Candidate lists: 4,000/10,000
  Candidate lists: 6,000/10,000
  Candidate lists: 8,000/10,000
  Candidate lists: 10,000/10,000
Candidate recall@50: 0.995200
Reference spectra selected: 370,125
Unique spectra needed: 380,125

[4/7] Loading selected MS/MS spectra...
  Scanned 500,000/2,539,608; loaded 116,136 selected spectra
  Scanned 1,000,000/2,539,608; loaded 23